In [0]:
catalog = "CineData_Analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# GOLD - DIM_MOVIES

## Regras de Negócio

A dimensão `dim_movies` consolida os principais atributos descritivos dos filmes, servindo como dimensão central do modelo analítico em esquema estrela (Star Schema).

### Origem dos Dados

A dimensão é construída a partir da tabela:

- silver.tb_info_filmes

### Seleção dos Atributos

Foram mantidos apenas os atributos relevantes para análise dimensional dos filmes.

A estrutura final é composta pelos seguintes campos:

- sk_movie_id
- id_filme
- titulo
- data_lancamento
- ano_lancamento
- duracao_minutos
- idioma_original
- status_filme
- sinopse

### Criação da Chave Substituta

- Foi criada a coluna `sk_movie_id` para atuar como chave primária da dimensão.
- A chave substituta permite independência em relação à chave natural da origem.
- A geração foi realizada utilizando identificadores únicos produzidos pelo Spark.

### Tratamento da Duração

- A coluna `duracao_minutos` foi convertida para tipo numérico inteiro.
- Valores incompatíveis com o formato esperado foram tratados como nulos.
- O tratamento evita falhas decorrentes de registros contaminados por problemas de qualidade na origem.

### Qualidade dos Dados

- Registros com valores inválidos em duração não interrompem o processamento.
- Os dados foram padronizados para suportar relacionamentos com tabelas fato e tabelas bridge da camada Gold.
- A dimensão preserva a chave natural `id_filme` para integração com as demais estruturas analíticas.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- sk_movie_id
- id_filme
- titulo
- data_lancamento
- ano_lancamento
- duracao_minutos
- idioma_original
- status_filme
- sinopse


In [0]:
from pyspark.sql import functions as F, Row

df_silver_filmes = spark.table(f"{silver_schema}.tb_info_filmes")

dim_movies = (
    df_silver_filmes

    .select("id_filme","titulo","data_lancamento","ano_lancamento","duracao_minutos","idioma_original","status_filme","sinopse")

    .withColumn("sk_movie_id",F.monotonically_increasing_id())
)

dim_movies = (
    dim_movies.select("sk_movie_id","id_filme","titulo","data_lancamento","ano_lancamento","duracao_minutos","idioma_original","status_filme","sinopse")
)

dim_movies = (
    dim_movies

    .withColumn("duracao_minutos",F.when(F.col("duracao_minutos").rlike("^[0-9]+$"),F.col("duracao_minutos").cast("int")))
)

dim_movies.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{gold_schema}.dim_movies"
    )

# GOLD - DIM_PEOPLE

## Regras de Negócio

A dimensão `dim_people` consolida todas as pessoas físicas envolvidas na produção dos filmes, disponibilizando uma estrutura única para análises relacionadas a elenco, direção e roteirização dentro do modelo dimensional (Star Schema).

### Origem dos Dados

A dimensão foi construída a partir da tabela:

- silver.tb_pessoas_empresas

### Seleção das Pessoas Físicas

A tabela de origem contém diferentes tipos de entidades associadas aos filmes.

Foram mantidas apenas entidades classificadas como:

- Ator
- Diretor
- Roteirista

Entidades classificadas como produtoras foram excluídas dessa dimensão, pois são tratadas separadamente na dimensão de empresas.

### Deduplicação das Pessoas

Uma mesma pessoa pode estar associada a diversos filmes na camada Silver.

Para construção da dimensão foi realizada a remoção de duplicidades utilizando a combinação:

- nome_pessoa
- tipo_pessoa

Esse tratamento garante que cada pessoa seja representada apenas uma única vez na dimensão.

Exemplo:

| Origem |
|---------|
| Ryan Reynolds \| Ator |
| Ryan Reynolds \| Ator |
| Ryan Reynolds \| Ator |

Transformado em:

| nome_pessoa | tipo_pessoa |
|------------|-------------|
| Ryan Reynolds | Ator |

### Padronização das Colunas

As colunas foram renomeadas para aderência ao modelo dimensional:

| Origem | Destino |
|---------|---------|
| entidade | nome_pessoa |
| tipo_entidade | tipo_pessoa |

### Criação da Chave Substituta

Foi criada a coluna:

- `sk_person_id`

A chave substituta atua como chave primária da dimensão e será utilizada nos relacionamentos com as tabelas bridge e fato da camada Gold.

A geração foi realizada por meio de identificadores únicos produzidos pelo Spark.

### Validações Realizadas

Foram realizadas as seguintes validações:

- Ausência de valores nulos em `sk_person_id`
- Ausência de chaves substitutas duplicadas
- Ausência de registros duplicados para a combinação:
  - nome_pessoa
  - tipo_pessoa

### Qualidade dos Dados

- Apenas pessoas físicas são mantidas na dimensão.
- Cada pessoa possui uma única chave substituta.
- Não existem registros duplicados após o processo de consolidação.
- A estrutura resultante está preparada para relacionamento com as tabelas bridge do modelo estrela.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- sk_person_id
- nome_pessoa
- tipo_pessoa

In [0]:
df_silver_pessoas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

dim_people = (
    df_silver_pessoas

    .filter(F.col("tipo_entidade").isin("Ator""Diretor","Roteirista"))
)

dim_people = (
    dim_people

    .select("entidade","tipo_entidade")

    .distinct()
)

dim_people = (
    dim_people

    .withColumnRenamed("entidade","nome_pessoa")

    .withColumnRenamed("tipo_entidade","tipo_pessoa")
)

dim_people = (
    dim_people

    .withColumn("sk_person_id",F.monotonically_increasing_id())
)

dim_people.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{gold_schema}.dim_people"
    )

# GOLD - DIM_COMPANIES

## Regras de Negócio

A dimensão `dim_companies` consolida o catálogo de produtoras e estúdios cinematográficos associados aos filmes, fornecendo uma estrutura única para relacionamentos analíticos no modelo dimensional (Star Schema).

### Origem dos Dados

A dimensão foi construída a partir da tabela:

- silver.tb_pessoas_empresas

### Seleção das Produtoras

A tabela de origem contém diferentes tipos de entidades relacionadas aos filmes.

Para a construção desta dimensão foram mantidas apenas as entidades classificadas como:

- Produtora

Foram descartadas as entidades classificadas como:

- Ator
- Diretor
- Roteirista

Dessa forma, a dimensão passa a representar exclusivamente empresas envolvidas na produção das obras.

### Deduplicação das Produtoras

Uma mesma produtora pode estar associada a diversos filmes na camada Silver.

Foi realizada a remoção de duplicidades mantendo apenas uma ocorrência de cada produtora.

Exemplo:

| Origem |
|---------|
| Warner Bros. Pictures |
| Warner Bros. Pictures |
| Warner Bros. Pictures |

Transformado em:

| nome_produtora |
|----------------|
| Warner Bros. Pictures |

### Padronização da Estrutura

A coluna de origem foi renomeada para aderência ao modelo dimensional:

| Origem | Destino |
|---------|---------|
| entidade | nome_produtora |

### Criação da Chave Substituta

Foi criada a coluna:

- `sk_company_id`

A chave substituta atua como chave primária da dimensão e será utilizada nos relacionamentos com tabelas bridge e tabelas fato da camada Gold.

A geração foi realizada por meio de identificadores únicos produzidos pelo Spark.

### Validações Realizadas

Foram realizadas as seguintes validações:

- Ausência de valores nulos em `sk_company_id`
- Ausência de chaves substitutas duplicadas
- Ausência de produtoras duplicadas após o processo de deduplicação

### Qualidade dos Dados

- Cada produtora é representada apenas uma única vez na dimensão.
- Todas as produtoras possuem uma chave substituta única.
- A dimensão fornece um catálogo consistente de empresas relacionadas aos filmes.
- A estrutura resultante está preparada para integração com tabelas bridge e tabelas fato do modelo estrela.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- sk_company_id
- nome_produtora

In [0]:
df_silver_pessoas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

dim_companies = (
    df_silver_pessoas

    .filter(F.col("tipo_entidade") == "Produtora")
)

dim_companies = (
    df_silver_pessoas

    .filter(F.col("tipo_entidade") == "Produtora")

    .select("entidade")

    .distinct()
)

dim_companies = (
    dim_companies

    .withColumnRenamed("entidade","nome_produtora")
)

dim_companies = (
    dim_companies

    .withColumn("sk_company_id",F.monotonically_increasing_id())
)

dim_companies.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{gold_schema}.dim_companies"
    )

# GOLD - DIM_GENRES

## Regras de Negócio

A dimensão `dim_genres` consolida o catálogo único de gêneros cinematográficos presentes nos filmes, fornecendo uma dimensão padronizada para classificação temática e análises de conteúdo dentro do modelo dimensional (Star Schema).

### Origem dos Dados

A dimensão foi construída a partir da tabela:

- silver.tb_generos

### Consolidação dos Gêneros

A tabela de origem já possui um registro por gênero associado ao filme.

Para construção da dimensão foi realizado o processo de consolidação dos gêneros em um catálogo único, mantendo apenas valores distintos.

Exemplo:

| Origem |
|---------|
| Action |
| Action |
| Action |
| Adventure |
| Adventure |

Transformado em:

| nome_genero |
|-------------|
| Action |
| Adventure |

### Deduplicação dos Registros

Foi aplicada a remoção de duplicidades para garantir que cada gênero seja representado apenas uma única vez na dimensão.

Esse processo assegura consistência e evita redundâncias nos relacionamentos com as tabelas bridge.

### Padronização da Estrutura

A coluna de origem foi renomeada para adequação ao modelo dimensional:

| Origem | Destino |
|---------|---------|
| genero | nome_genero |

### Criação da Chave Substituta

Foi criada a coluna:

- `sk_genre_id`

A chave substituta atua como chave primária da dimensão e será utilizada nos relacionamentos com tabelas bridge e tabelas fato da camada Gold.

A geração foi realizada utilizando identificadores únicos produzidos pelo Spark.

### Validações Realizadas

Foram realizadas as seguintes validações:

- Ausência de valores nulos em `sk_genre_id`
- Ausência de chaves substitutas duplicadas
- Ausência de gêneros duplicados após a consolidação

### Qualidade dos Dados

- Cada gênero é representado apenas uma única vez na dimensão.
- Todos os gêneros possuem uma chave substituta única.
- A estrutura está preparada para integração com as tabelas bridge do modelo estrela.
- A dimensão fornece um catálogo consistente para análises temáticas e segmentação de conteúdo.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- sk_genre_id
- nome_genero

In [0]:
df_silver_generos = spark.table(f"{silver_schema}.tb_generos")

dim_genres = (
    df_silver_generos

    .select("genero")

    .distinct()
)

dim_genres = (
    dim_genres

    .withColumnRenamed("genero","nome_genero")
)

dim_genres = (
    dim_genres

    .withColumn("sk_genre_id",F.monotonically_increasing_id())
)

dim_genres.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{gold_schema}.dim_genres"
    )



# GOLD - DIM_REVIEWS

## Regras de Negócio

A dimensão `dim_reviews` consolida os indicadores de avaliação dos usuários para cada filme, disponibilizando métricas agregadas para análises de satisfação e percepção do público dentro do modelo dimensional (Star Schema).

### Origem dos Dados

A dimensão foi construída a partir da tabela:

- silver.tb_avaliacoes_usuarios

### Consolidação das Avaliações

A origem contém múltiplas avaliações para um mesmo filme.

Para construção da dimensão, as avaliações foram agregadas por:

- id_filme

Gerando uma única linha por filme.

### Métricas Calculadas

Foram calculados os seguintes indicadores:

#### Quantidade de Avaliações

Quantidade total de avaliações realizadas pelos usuários para cada filme.

```text
qtd_avaliacoes = COUNT(*)
```

#### Nota Média dos U*uários

Representa a média das not*s atribuídas pelos usuários ao fil*e.

```text
nota_media_usuario = A*G(nota_usuario)
```

### Estrutura*Agregada

Exemplo:

| id_filme | q*d_avaliacoes | nota_media_usuario *
|-----------|----------------|---*----------------|
| 447163 | 3 | 6*2 |
| 442113 | 2 | 4.2 |

### Cria*ão da Ch*ve Substituta

Foi criada a coluna*

- `sk_review_id`

A chave substi*uta atua como chave primária da di*ensão e será utilizada nos relacio*amentos com tabelas fato da camada*Gold.

A geração foi realizada uti*izando identificadores únicos prod*zidos pelo Spark.

### Validações *ealizadas

Foram realizadas as seg*intes validações:

- Ausência de v*lores nulos em `sk_review_id`
- Au*ência de chaves substitutas duplic*das
- Garantia de apenas um regist*o por filme

### Qualidade dos Dad*s

- Cada filme possui apenas uma *inha na dimensão.
- Todos os regis*ros possuem chave substituta válid*.
- Os indicadores agregados permi*em análises de aceitação e satisfa*ão dos usuários.
- A dimensão está*preparada para integração com a ta*ela fato principal do modelo estre*a.

### Estrutura Final da Tabela
*A tabela é composta pelas seguinte* colunas:

- sk_review_id
- id_fil*e
- qtd_avaliacoes
- nota_media_us*ario

In [0]:
df_silver_reviews = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")

dim_reviews = (
    df_silver_reviews

    .groupBy("id_filme")

    .agg(F.count("*").alias("qtd_avaliacoes"),F.avg("nota_usuario").alias("nota_media_usuario"))
)

dim_reviews = (
    dim_reviews

    .withColumn("sk_review_id",F.monotonically_increasing_id())
)

dim_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{gold_schema}.dim_reviews"
    )

# GOLD - BRIDGE_MOVIE_GENRE

## Regras de Negócio

A tabela `bridge_movie_genre` representa o relacionamento entre filmes e gêneros dentro do modelo dimensional (Star Schema).

### Objetivo

Um mesmo filme pode estar associado a vários gêneros e um mesmo gênero pode estar associado a diversos filmes.

Essa relação caracteriza um relacionamento muitos-para-muitos (N:N), exigindo a utilização de uma tabela bridge para sua representação.

### Origem dos Dados

A bridge foi construída a partir das tabelas:

- gold.dim_movies
- gold.dim_genres
- silver.tb_generos

### Resolução das Chaves

A tabela de origem contém:

| Campo |
|---------|
| id_filme |
| genero |

Para adequação ao modelo dimensional, foram resolvidas as chaves substitutas das respectivas dimensões:

#### Filme

```text
id_filme
↓
sk_movie_id
```

#### Gênero

```text
genero
↓
sk_genre_id
```

### Relacionamento Muitos-para-Muitos

Exemplo:

| Filme | Gênero |
|---------|---------|
| Deadpool | Action |
| Deadpool | Adventure |
| Deadpool | Comedy |

Transformado em:

| sk_movie_id | sk_genre_id |
|---------|---------|
| 336 | 10 |
| 336 | 12 |
| 336 | 2 |

### Tratamento de Duplicidades

Foi aplicada a remoção de duplicidades para garantir que cada relacionamento entre filme e gênero seja representado apenas uma única vez.

A unicidade é garantida pela combinação:

- sk_movie_id
- sk_genre_id

### Validações Realizadas

Foram realizadas as seguintes validações:

- Ausência de valores nulos em `sk_movie_id`
- Ausência de valores nulos em `sk_genre_id`
- Ausência de relacionamentos duplicados

### Qualidade dos Dados

- Todos os filmes possuem referência válida para a dimensão de filmes.
- Todos os gêneros possuem referência válida para a dimensão de gêneros.
- Os relacionamentos foram normalizados para suportar análises multidimensionais.
- A estrutura está preparada para utilização conjunta com a tabela fato principal da camada Gold.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- sk_movie_id
- sk_genre_id

In [0]:
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

df_dim_genres = spark.table(f"{gold_schema}.dim_genres")

df_silver_generos = spark.table(f"{silver_schema}.tb_generos")

bridge_movie_genre = (
    df_silver_generos

    .join(df_dim_movies.select("id_filme","sk_movie_id"),on="id_filme",how="inner")
)

bridge_movie_genre = (
    bridge_movie_genre

    .join(df_dim_genres.select("sk_genre_id","nome_genero"),bridge_movie_genre["genero"] == df_dim_genres["nome_genero"],"inner")
)

bridge_movie_genre = (
    bridge_movie_genre

    .select("sk_movie_id","sk_genre_id")

    .dropDuplicates()
)

bridge_movie_genre.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{gold_schema}.bridge_movie_genre"
    )

# GOLD - BRIDGE_MOVIE_PERSON

## Regras de Negócio

A tabela `bridge_movie_person` representa o relacionamento entre filmes e pessoas físicas dentro do modelo dimensional (Star Schema).

### Objetivo

Um mesmo filme pode possuir diversos participantes e uma mesma pessoa pode estar associada a diversos filmes.

Essa relação caracteriza um relacionamento muitos-para-muitos (N:N), exigindo a utilização de uma tabela bridge para sua representação.

### Origem dos Dados

A bridge foi construída a partir das tabelas:

- gold.dim_movies
- gold.dim_people
- silver.tb_pessoas_empresas

### Seleção das Pessoas

Foram consideradas apenas entidades classificadas como:

- Ator
- Diretor
- Roteirista

Entidades classificadas como produtoras foram excluídas desta bridge, pois possuem relacionamento próprio por meio da tabela `bridge_movie_company`.

### Resolução das Chaves

A tabela de origem contém:

| Campo |
|---------|
| id_filme |
| entidade |
| tipo_entidade |

Para adequação ao modelo dimensional, foram resolvidas as chaves substitutas das respectivas dimensões.

#### Filme

```text
id_filme
↓
sk_movie_id
```

#### Pessoa

```text
entidade + tipo_entidade
↓
sk_person_id
```

### Relacionamento Muitos-para-Muitos

Exemplo:

| Filme | Pessoa | Tipo |
|---------|---------|---------|
| Avengers: Endgame | Chris Hemsworth | Ator |
| Avengers: Endgame | Anthony Russo | Diretor |
| Avengers: Endgame | Joe Russo | Diretor |

Transformado em:

| sk_movie_id | sk_person_id |
|---------|---------|
| 384 | 1254 |
| 384 | 2241 |
| 384 | 2242 |

### Tratamento de Duplicidades

Foi aplicada a remoção de duplicidades para garantir que cada relacionamento entre filme e pessoa seja representado apenas uma única vez.

A unicidade é garantida pela combinação:

- sk_movie_id
- sk_person_id

### Validações Realizadas

Foram realizadas as seguintes validações:

- Ausência de valores nulos em `sk_movie_id`
- Ausência de valores nulos em `sk_person_id`
- Ausência de relacionamentos duplicados

### Qualidade dos Dados

- Todos os filmes possuem referência válida para a dimensão de filmes.
- Todas as pessoas possuem referência válida para a dimensão de pessoas.
- Os relacionamentos foram normalizados para suportar análises multidimensionais.
- A estrutura está preparada para utilização conjunta com a tabela fato principal da camada Gold.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- sk_movie_id
- sk_person_id

In [0]:
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

df_dim_people = spark.table(f"{gold_schema}.dim_people")

df_silver_pessoas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

bridge_movie_person = (
    df_silver_pessoas

    .filter(F.col("tipo_entidade").isin("Ator","Diretor","Roteirista"))
)

bridge_movie_person = (
    bridge_movie_person

    .join(df_dim_movies.select("id_filme","sk_movie_id"),on="id_filme",how="inner")
)

bridge_movie_person = (
    bridge_movie_person

    .join(df_dim_people.select("sk_person_id","nome_pessoa","tipo_pessoa"),((bridge_movie_person["entidade"] == df_dim_people["nome_pessoa"]) &(bridge_movie_person["tipo_entidade"] == df_dim_people["tipo_pessoa"])),"inner")
)

bridge_movie_person = (
    bridge_movie_person

    .select("sk_movie_id","sk_person_id")

    .dropDuplicates()
)

bridge_movie_person.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{gold_schema}.bridge_movie_person"
    )

# GOLD - BRIDGE_MOVIE_COMPANY

## Regras de Negócio

A tabela `bridge_movie_company` representa o relacionamento entre filmes e produtoras dentro do modelo dimensional (Star Schema).

### Objetivo

Um mesmo filme pode estar associado a diversas produtoras e uma mesma produtora pode participar da produção de diversos filmes.

Essa relação caracteriza um relacionamento muitos-para-muitos (N:N), exigindo a utilização de uma tabela bridge para sua representação.

### Origem dos Dados

A bridge foi construída a partir das tabelas:

- gold.dim_movies
- gold.dim_companies
- silver.tb_pessoas_empresas

### Seleção das Produtoras

Foram consideradas apenas entidades classificadas como:

- Produtora

Foram descartadas entidades classificadas como:

- Ator
- Diretor
- Roteirista

### Resolução das Chaves

A tabela de origem contém:

| Campo |
|---------|
| id_filme |
| entidade |

Para adequação ao modelo dimensional, foram resolvidas as chaves substitutas das respectivas dimensões.

#### Filme

```text
id_filme
↓
sk_movie_id
```

#### Produtora

```text
entidade
↓
sk_company_id
```

### Relacionamento Muitos-para-Muitos

Exemplo:

| Filme | Produtora |
|---------|---------|
| Deadpool | 20th Century Fox |
| Deadpool | Maximum Effort |
| Deadpool | Genre Films |

Transformado em:

| sk_movie_id | sk_company_id |
|---------|---------|
| 336 | 102 |
| 336 | 215 |
| 336 | 489 |

### Tratamento de Duplicidades

Foi aplicada a remoção de duplicidades para garantir que cada relacionamento entre filme e produtora seja representado apenas uma única vez.

A unicidade é garantida pela combinação:

- sk_movie_id
- sk_company_id

### Validações Realizadas

Foram realizadas as seguintes validações:

- Ausência de valores nulos em `sk_movie_id`
- Ausência de valores nulos em `sk_company_id`
- Ausência de relacionamentos duplicados

### Qualidade dos Dados

- Todos os filmes possuem referência válida para a dimensão de filmes.
- Todas as produtoras possuem referência válida para a dimensão de produtoras.
- Os relacionamentos foram normalizados para suportar análises multidimensionais.
- A estrutura encontra-se preparada para utilização conjunta com a tabela fato principal da camada Gold.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- sk_movie_id
- sk_company_id

In [0]:
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

df_dim_companies = spark.table(f"{gold_schema}.dim_companies")

df_silver_pessoas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

bridge_movie_company = (
    df_silver_pessoas

    .filter(F.col("tipo_entidade") == "Produtora")
)

bridge_movie_company = (
    bridge_movie_company

    .join(df_dim_movies.select("id_filme","sk_movie_id"),on="id_filme",how="inner")
)

bridge_movie_company = (
    bridge_movie_company

    .join(
        df_dim_companies.select("sk_company_id","nome_produtora"),bridge_movie_company["entidade"] == df_dim_companies["nome_produtora"],"inner")
)

bridge_movie_company = (
    bridge_movie_company

    .select("sk_movie_id","sk_company_id")

    .dropDuplicates()
)

bridge_movie_company.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{gold_schema}.bridge_movie_company"
    )

# GOLD - FACT_MOVIES_PERFORMANCE

## Regras de Negócio

A tabela `fact_movies_performance` consolida os principais indicadores de desempenho dos filmes, reunindo métricas financeiras, métricas de engajamento e avaliações dos usuários em uma única estrutura analítica.

Esta tabela representa a fato principal do modelo dimensional (Star Schema).

### Origem dos Dados

A fato foi construída a partir das tabelas:

- gold.dim_movies
- gold.dim_reviews
- silver.tb_financeiro_filmes
- silver.tb_metricas_engajamento

### Granularidade

A tabela possui granularidade:

```text
1 filme = 1 registro
```

Cada linha representa o desempenho consolidado de um único filme.

### Integração das Dimensões

Foram incorporadas as chaves substitutas das dimensões:

#### Filme

```text
id_filme
↓
sk_movie_id
```

#### Avaliações

```text
id_filme
↓
sk_review_id
```

### Métricas Financeiras

Foram incorporadas as métricas provenientes da tabela financeira:

- orcamento_usd
- receita_usd
- orcamento_brl
- receita_brl
- lucro_usd
- lucro_brl
- margem_lucro_pct

Esses indicadores permitem análises de rentabilidade, retorno financeiro e lucratividade dos filmes.

### Métricas de Engajamento

Foram incorporadas as métricas provenientes das plataformas TMDB e IMDb:

- popularidade
- nota_media_tmdb
- qtd_votos_tmdb
- nota_media_imdb
- qtd_votos_imdb

Esses indicadores permitem análises de popularidade e aceitação dos filmes pelo público.

### Métricas de Avaliação dos Usuários

Foram incorporadas as métricas agregadas da dimensão de avaliações:

- qtd_avaliacoes
- nota_media_usuario

Esses indicadores representam o comportamento e a percepção dos usuários em relação aos filmes.

### Tratamento de Duplicidades

Durante o processo de integração foram identificadas duplicidades nas tabelas analíticas da camada Silver.

Para garantir a granularidade correta da fato:

```text
1 filme = 1 registro
```

foram mantidos apenas registros únicos por `id_filme` antes da realização dos joins.

### Validações Realizadas

Foram realizadas as seguintes validações:

- Ausência de duplicidade por `sk_movie_id`
- Garantia de uma única linha por filme
- Integridade das chaves substitutas utilizadas nos relacionamentos
- Consistência dos joins entre dimensões e tabelas analíticas

### Qualidade dos Dados

- A tabela concentra indicadores financeiros, operacionais e de engajamento.
- Cada filme possui um único registro consolidado.
- A estrutura está preparada para análises analíticas e construção de dashboards.
- A modelagem segue o padrão dimensional de tabela fato conectado às dimensões do Star Schema.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- sk_movie_id
- sk_review_id
- orcamento_usd
- receita_usd
- orcamento_brl
- receita_brl
- lucro_usd
- lucro_brl
- margem_lucro_pct
- popularidade
- nota_media_tmdb
- qtd_votos_tmdb
- nota_media_imdb
- qtd_votos_imdb
- qtd_avaliacoes
- nota_media_usuario

In [0]:
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

df_dim_reviews = spark.table(f"{gold_schema}.dim_reviews")

df_financeiro = (
    spark.table(f"{silver_schema}.tb_financeiro_filmes")
    .dropDuplicates(["id_filme"])
)

df_metricas = (spark.table(f"{silver_schema}.tb_metricas_engajamento")
    .dropDuplicates(["id_filme"])
)


fact_movies_performance = (
    df_dim_movies

    .join(df_dim_reviews.select("id_filme","sk_review_id","qtd_avaliacoes","nota_media_usuario"),on="id_filme",how="left")
)

fact_movies_performance = (
    fact_movies_performance

    .join(df_financeiro.select("id_filme","orcamento_usd","receita_usd","orcamento_brl","receita_brl","lucro_usd","lucro_brl","margem_lucro_pct"),on="id_filme",how="left")
)

fact_movies_performance = (
    fact_movies_performance

    .join(df_metricas.select("id_filme","popularidade","nota_media_tmdb","qtd_votos_tmdb","nota_media_imdb","qtd_votos_imdb"),on="id_filme",how="left")
)

fact_movies_performance = (
    fact_movies_performance

    .select("sk_movie_id","sk_review_id","orcamento_usd","receita_usd","orcamento_brl","receita_brl","lucro_usd","lucro_brl","margem_lucro_pct","popularidade","nota_media_tmdb","qtd_votos_tmdb","nota_media_imdb","qtd_votos_imdb","qtd_avaliacoes","nota_media_usuario")
)

df_metricas_gold = (
    df_metricas
    .dropDuplicates(["id_filme"])
)

fact_movies_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{gold_schema}.fact_movies_performance"
    )